<a href="https://colab.research.google.com/github/asadovkamran/aidl_nlp_projects/blob/main/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tatoeba (English-French)

In [ ]:
import torch
import re
import pandas as pd
from torch.utils.data import Dataset

In [ ]:
# --- STEP 1: THE DATA CLEANER ---
def clean_text(text):
    """Standardizes text: lowercase, punctuation spacing, and noise removal."""
    text = str(text).lower().strip()
    # Separate punctuation so it becomes its own token: "student." -> "student ."
    text = re.sub(r"([.!?])", r" \1", text)
    # Keep only letters and basic punctuation
    text = re.sub(r"[^a-zA-Z.!?]+", r" ", text)
    return text.strip()

In [ ]:
# --- STEP 2: THE BOUNCER (FILTERING) ---
def filter_pairs(pairs, max_length=15):
    """Requirement: Keeps only sentences with <= 15 tokens."""
    return [p for p in pairs if len(p[0].split()) <= max_length and len(p[1].split()) <= max_length]

In [ ]:
# --- STEP 3: THE DICTIONARY (VOCABULARY CONSTRUCTION) ---
class Vocab:
    def __init__(self):
        # Mandatory Special Tokens
        self.word2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.index2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        # TOKENIZATION: Splitting strings into individual words
        for word in sentence.split():
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1

In [ ]:
# --- STEP 4: THE DATASET HANDLER ---
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_sent, trg_sent = self.pairs[idx]
        # NUMERICALIZATION: Turning words into ID numbers
        # Source gets <EOS> (2)
        src_ids = [self.src_vocab.word2index.get(w, 3) for w in src_sent.split()] + [2]
        # Target gets <SOS> (1) and <EOS> (2)
        trg_ids = [1] + [self.trg_vocab.word2index.get(w, 3) for w in trg_sent.split()] + [2]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

In [ ]:
# --- STEP 5: THE MASTER PIPELINE (RELATIONAL JOIN) ---
def run_person_d_pipeline():
    print("--- STARTING PERSON D PIPELINE ---")

    try:
        # 1. Load the raw files you uploaded
        print("1. Loading raw .tsv and .csv files...             ... 🐞🌱")
        eng_df = pd.read_csv('eng_sentences.tsv', sep='\t', names=['id', 'lang', 'text'], usecols=[0, 2])
        fra_df = pd.read_csv('fra_sentences.tsv', sep='\t', names=['id', 'lang', 'text'], usecols=[0, 2])
        links_df = pd.read_csv('links.csv', sep='\t', names=['eng_id', 'fra_id'])

        # 2. Relational Join: Linking English to French via ID
        print("2. Joining tables using ID links...               ... 🐛🌱")
        merged = links_df.merge(eng_df, left_on='eng_id', right_on='id')
        merged = merged.rename(columns={'text': 'english'})

        final_df = merged.merge(fra_df, left_on='fra_id', right_on='id')
        final_df = final_df.rename(columns={'text': 'french'})

        # Convert to list of strings
        raw_pairs = final_df[['english', 'french']].values.tolist()

        # 3. Clean and Filter
        print(f"3. Cleaning and filtering {len(raw_pairs)} pairs...         ... 🦋🌱")
        cleaned_pairs = [(clean_text(eng), clean_text(fra)) for eng, fra in raw_pairs]
        filtered_pairs = filter_pairs(cleaned_pairs, max_length=15)

        # 4. Limit for training and build Vocab
        dataset_subset = filtered_pairs[:20000] # Adjust based on your RAM
        src_vocab = Vocab()
        trg_vocab = Vocab()

        for eng, fra in dataset_subset:
            src_vocab.add_sentence(eng)
            trg_vocab.add_sentence(fra)

        print(f"\n--- SUCCESS REPORT ---")
        print(f"Total Linked Pairs Found: {len(raw_pairs)}")
        print(f"Pairs Meeting 15-Token Limit: {len(filtered_pairs)}")
        print(f"Final Dataset Size: {len(dataset_subset)}")
        print(f"English Vocab Size (INPUT_DIM): {src_vocab.n_words}")
        print(f"French Vocab Size (OUTPUT_DIM): {trg_vocab.n_words}")

        return dataset_subset, src_vocab, trg_vocab

    except FileNotFoundError as e:
        print(f"ERROR: Could not find the files. Make sure eng_sentences.tsv, fra_sentences.tsv, and links.csv are uploaded. Details: {e}")
        return None, None, None

In [ ]:
# --- FINAL EXECUTION ---
pairs, src_vocab, trg_vocab = run_person_d_pipeline()

# Proof of work for the presentation
if pairs:
    example_eng, example_fra = pairs[0]
    print(f"\nExample Sentence 1 (Cleaned):")
    print(f"EN: {example_eng}")
    print(f"FR: {example_fra}")

--- STARTING PERSON D PIPELINE ---
1. Loading raw .tsv and .csv files...             ... 🐞🌱
2. Joining tables using ID links...               ... 🐛🌱
3. Cleaning and filtering 433875 pairs...         ... 🦋🌱

--- SUCCESS REPORT ---
Total Linked Pairs Found: 433875
Pairs Meeting 15-Token Limit: 410506
Final Dataset Size: 20000
English Vocab Size (INPUT_DIM): 6969
French Vocab Size (OUTPUT_DIM): 9215

Example Sentence 1 (Cleaned):
EN: let s try something .
FR: essayons quelque chose !


- we have 433,875 English sentences matched with 433,875 French translations

- then we got that 94% of your data was "short enough" with at most 15 tokens - 410,506 sentences

- then we chose "representative sample" with 20,000 sentences

- and those (6,969 and 9,215) are unique words (tokens) from those 20,000 sentences

- the encoder layer must be exactly 6,969 units wide so it can recognize every english word in your dictionary

- the decoder layer must be exactly 9,215 units wide so it can choose between all the possible french words when it translates

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import random
import math
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# --- Model dimensions ---
INPUT_DIM  = src_vocab.n_words   # English vocab size
OUTPUT_DIM = trg_vocab.n_words   # French vocab size
ENC_EMB_DIM   = 256
DEC_EMB_DIM   = 256
HID_DIM       = 512              # encoder & decoder hidden size
ATTN_DIM      = 64               # Bahdanau alignment layer size

# --- Training ---
BATCH_SIZE    = 64
N_EPOCHS      = 15
CLIP          = 1.0              # gradient clipping
TEACHER_FORCE = 0.5              # probability of using ground truth token
LEARNING_RATE = 1e-3
PAD_IDX       = 0

# --- Inference ---
MAX_LEN = 20                     # max tokens to generate during translation

print("Hyperparameters set.")

Hyperparameters set.


In [ ]:
from torch.utils.data import Dataset

# Re-use TranslationDataset class
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, trg_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_sent, trg_sent = self.pairs[idx]
        src_ids = [self.src_vocab.word2index.get(w, 3) for w in src_sent.split()] + [2]
        trg_ids = [1] + [self.trg_vocab.word2index.get(w, 3) for w in trg_sent.split()] + [2]
        return torch.tensor(src_ids), torch.tensor(trg_ids)


def collate_fn(batch):
    """Pads sequences in a batch to the same length."""
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=False, padding_value=PAD_IDX)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, batch_first=False, padding_value=PAD_IDX)
    return src_padded, trg_padded


full_dataset = TranslationDataset(pairs, src_vocab, trg_vocab)

n_total = len(full_dataset)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)
n_test  = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")

Train: 16000 | Val: 2000 | Test: 2000


## Model Architecture

In [ ]:
#Encoder (shared by both Vanilla and Attention models)

class Encoder(nn.Module):
    """
    GRU encoder.
    Reads the source sentence token-by-token and outputs:
      - all hidden states  (seq_len, batch, hid_dim)  ← needed for attention
      - final hidden state (1, batch, hid_dim)        ← used to init decoder
    """
    def __init__(self, input_dim, emb_dim, hid_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=PAD_IDX)
        self.gru       = nn.GRU(emb_dim, hid_dim, batch_first=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        # src: (src_len, batch)
        embedded = self.dropout(self.embedding(src))   # (src_len, batch, emb_dim)
        outputs, hidden = self.gru(embedded)           # outputs: (src_len, batch, hid_dim)
                                                       # hidden:  (1, batch, hid_dim)
        return outputs, hidden

In [ ]:
#Vanilla Decoder (no attention — baseline)

class VanillaDecoder(nn.Module):
    """
    Standard RNN decoder. Uses ONLY the encoder's final hidden state
    as context — the fixed-length bottleneck that attention fixes.
    """
    def __init__(self, output_dim, emb_dim, hid_dim, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        self.gru        = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=False)
        self.fc_out     = nn.Linear(hid_dim, output_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, trg_token, hidden, encoder_outputs):
        # trg_token: (batch,)  — one token at a time
        # hidden:    (1, batch, hid_dim)
        # encoder_outputs: (src_len, batch, hid_dim) — NOT used here (vanilla)

        trg_token = trg_token.unsqueeze(0)                    # (1, batch)
        embedded  = self.dropout(self.embedding(trg_token))   # (1, batch, emb_dim)

        # Context = encoder's last hidden state (fixed vector)
        context  = hidden                                     # (1, batch, hid_dim)
        rnn_input = torch.cat([embedded, context], dim=2)    # (1, batch, emb+hid)

        output, hidden = self.gru(rnn_input, hidden)         # output: (1, batch, hid)
        prediction = self.fc_out(output.squeeze(0))          # (batch, output_dim)
        return prediction, hidden, None                      # None = no attention weights

In [ ]:
#Bahdanau Attention Mechanism

class BahdanauAttention(nn.Module):
    """
    Additive (Bahdanau) attention.

    Score function:
        e_{t,i} = v_a · tanh( W_a · s_t  +  U_a · h_i )

    where:
        s_t = current decoder hidden state
        h_i = encoder hidden state at position i
        W_a, U_a, v_a = learnable parameters

    Returns:
        context:  (batch, hid_dim)   — weighted sum of encoder states
        weights:  (batch, src_len)   — attention distribution (for visualization)
    """
    def __init__(self, hid_dim, attn_dim):
        super().__init__()
        self.W_a = nn.Linear(hid_dim, attn_dim, bias=False)   # transforms decoder state
        self.U_a = nn.Linear(hid_dim, attn_dim, bias=False)   # transforms encoder states
        self.v_a = nn.Linear(attn_dim, 1, bias=False)         # score projection

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden:  (1, batch, hid_dim)  →  (batch, 1, hid_dim)
        # encoder_outputs: (src_len, batch, hid_dim) → (batch, src_len, hid_dim)

        dec_hid = decoder_hidden.permute(1, 0, 2)             # (batch, 1, hid_dim)
        enc_out = encoder_outputs.permute(1, 0, 2)            # (batch, src_len, hid_dim)

        # Broadcast: W_a·s_t is (batch,1,attn_dim), U_a·h_i is (batch,src_len,attn_dim)
        energy = torch.tanh(
            self.W_a(dec_hid) + self.U_a(enc_out)            # (batch, src_len, attn_dim)
        )
        scores = self.v_a(energy).squeeze(2)                  # (batch, src_len)

        weights = F.softmax(scores, dim=1)                    # (batch, src_len)  sums to 1

        # Weighted sum of encoder outputs
        context = torch.bmm(
            weights.unsqueeze(1),   # (batch, 1, src_len)
            enc_out                 # (batch, src_len, hid_dim)
        ).squeeze(1)                # (batch, hid_dim)

        return context, weights

In [ ]:
#Attention Decoder

class AttentionDecoder(nn.Module):
    """
    GRU decoder with Bahdanau attention.
    At each step, computes a dynamic context vector from ALL encoder states.
    """
    def __init__(self, output_dim, emb_dim, hid_dim, attn_dim, dropout=0.3):
        super().__init__()
        self.attention  = BahdanauAttention(hid_dim, attn_dim)
        self.embedding  = nn.Embedding(output_dim, emb_dim, padding_idx=PAD_IDX)
        # GRU input = embedding + context vector
        self.gru        = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=False)
        self.fc_out     = nn.Linear(hid_dim + hid_dim + emb_dim, output_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, trg_token, hidden, encoder_outputs):
        # trg_token:       (batch,)
        # hidden:          (1, batch, hid_dim)
        # encoder_outputs: (src_len, batch, hid_dim)

        trg_token = trg_token.unsqueeze(0)                     # (1, batch)
        embedded  = self.dropout(self.embedding(trg_token))    # (1, batch, emb_dim)

        # Compute attention context
        context, attn_weights = self.attention(hidden, encoder_outputs)
        context = context.unsqueeze(0)                         # (1, batch, hid_dim)

        # Feed embedding + context into GRU
        rnn_input = torch.cat([embedded, context], dim=2)      # (1, batch, emb+hid)
        output, hidden = self.gru(rnn_input, hidden)           # output: (1, batch, hid)

        # Final prediction from output + context + embedding (rich signal)
        combined   = torch.cat(
            [output.squeeze(0), context.squeeze(0), embedded.squeeze(0)], dim=1
        )                                                       # (batch, hid+hid+emb)
        prediction = self.fc_out(combined)                     # (batch, output_dim)

        return prediction, hidden, attn_weights

In [ ]:
#Seq2Seq Wrapper (works for both Vanilla and Attention)

class Seq2Seq(nn.Module):
    """
    Wraps Encoder + Decoder.
    teacher_forcing_ratio: probability of feeding ground truth instead of prediction.
    """
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (src_len, batch)
        # trg: (trg_len, batch)  — includes <SOS> at position 0

        trg_len, batch_size = trg.shape
        output_dim = self.decoder.fc_out.out_features

        # Tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, output_dim).to(src.device)

        # Encode the source sentence
        encoder_outputs, hidden = self.encoder(src)

        # First decoder input is <SOS>
        dec_input = trg[0, :]   # (batch,)

        for t in range(1, trg_len):
            prediction, hidden, _ = self.decoder(dec_input, hidden, encoder_outputs)
            outputs[t] = prediction

            # Teacher forcing decision
            use_teacher = random.random() < teacher_forcing_ratio
            dec_input = trg[t] if use_teacher else prediction.argmax(1)

        return outputs

## Initialize Both Models

In [ ]:
def init_weights(m):
    """Xavier uniform for linear & embedding layers."""
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ── Vanilla Seq2Seq ──────────────────────────────────────────────────────────
vanilla_enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM).to(device)
vanilla_dec = VanillaDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM).to(device)
vanilla_model = Seq2Seq(vanilla_enc, vanilla_dec).to(device)
vanilla_model.apply(init_weights)

# ── Attention Seq2Seq ────────────────────────────────────────────────────────
attn_enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM).to(device)
attn_dec = AttentionDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, ATTN_DIM).to(device)
attn_model = Seq2Seq(attn_enc, attn_dec).to(device)
attn_model.apply(init_weights)

print(f"Vanilla model parameters: {count_params(vanilla_model):,}")
print(f"Attention model parameters: {count_params(attn_model):,}")

Vanilla model parameters: 12,022,271
Attention model parameters: 19,164,991


## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_epoch(model, loader, optimizer, clip, teacher_forcing_ratio):
    model.train()
    epoch_loss = 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        output = model(src, trg, teacher_forcing_ratio)   # (trg_len, batch, output_dim)

        # Reshape for loss: skip <SOS> at position 0
        output_dim = output.shape[-1]
        output_flat = output[1:].reshape(-1, output_dim)  # ((trg_len-1)*batch, output_dim)
        trg_flat    = trg[1:].reshape(-1)                 # ((trg_len-1)*batch,)

        loss = criterion(output_flat, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate_epoch(model, loader):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)   # no teacher forcing at eval
            output_dim = output.shape[-1]
            output_flat = output[1:].reshape(-1, output_dim)
            trg_flat    = trg[1:].reshape(-1)
            loss = criterion(output_flat, trg_flat)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)


def run_training(model, model_name, save_path):
    optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')

    print(f"\n{'='*50}")
    print(f" Training: {model_name}")
    print(f"{'='*50}")

    for epoch in range(N_EPOCHS):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, CLIP, TEACHER_FORCE)
        val_loss   = evaluate_epoch(model, val_loader)
        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        elapsed = time.time() - t0
        print(f"Epoch {epoch+1:02}/{N_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} (PPL: {math.exp(train_loss):7.3f}) | "
              f"Val Loss: {val_loss:.4f} (PPL: {math.exp(val_loss):7.3f}) | "
              f"{elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"  ✔ Best model saved → {save_path}")

    return train_losses, val_losses

print("Training functions ready.")

Training functions ready.


## Train Both Models

In [ ]:
vanilla_train_losses, vanilla_val_losses = run_training(
    vanilla_model, "Vanilla Seq2Seq", "vanilla_best.pt"
)


 Training: Vanilla Seq2Seq
Epoch 01/15 | Train Loss: 5.5164 (PPL: 248.735) | Val Loss: 5.2020 (PPL: 181.644) | 435.9s
  ✔ Best model saved → vanilla_best.pt
Epoch 02/15 | Train Loss: 4.5678 (PPL:  96.334) | Val Loss: 4.8789 (PPL: 131.481) | 430.5s
  ✔ Best model saved → vanilla_best.pt
Epoch 03/15 | Train Loss: 4.0206 (PPL:  55.736) | Val Loss: 4.7015 (PPL: 110.115) | 431.6s
  ✔ Best model saved → vanilla_best.pt
Epoch 04/15 | Train Loss: 3.5771 (PPL:  35.771) | Val Loss: 4.5665 (PPL:  96.204) | 439.9s
  ✔ Best model saved → vanilla_best.pt
Epoch 05/15 | Train Loss: 3.1628 (PPL:  23.637) | Val Loss: 4.4581 (PPL:  86.326) | 436.6s
  ✔ Best model saved → vanilla_best.pt
Epoch 06/15 | Train Loss: 2.7844 (PPL:  16.189) | Val Loss: 4.3991 (PPL:  81.381) | 447.1s
  ✔ Best model saved → vanilla_best.pt
Epoch 07/15 | Train Loss: 2.4531 (PPL:  11.625) | Val Loss: 4.3821 (PPL:  80.003) | 432.9s
  ✔ Best model saved → vanilla_best.pt
Epoch 08/15 | Train Loss: 2.1774 (PPL:   8.823) | Val Loss: 4.

In [ ]:
attn_train_losses, attn_val_losses = run_training(
    attn_model, "Seq2Seq + Bahdanau Attention", "attention_best.pt"
)


 Training: Seq2Seq + Bahdanau Attention
Epoch 01/15 | Train Loss: 5.2563 (PPL: 191.769) | Val Loss: 5.0241 (PPL: 152.038) | 842.7s
  ✔ Best model saved → attention_best.pt
Epoch 02/15 | Train Loss: 4.1472 (PPL:  63.254) | Val Loss: 4.4519 (PPL:  85.792) | 839.5s
  ✔ Best model saved → attention_best.pt
Epoch 03/15 | Train Loss: 3.2765 (PPL:  26.484) | Val Loss: 4.0720 (PPL:  58.677) | 879.6s
  ✔ Best model saved → attention_best.pt
Epoch 04/15 | Train Loss: 2.5887 (PPL:  13.313) | Val Loss: 3.8424 (PPL:  46.637) | 843.9s
  ✔ Best model saved → attention_best.pt


## Learning Curves

In [ ]:
def plot_learning_curves(vanilla_train, vanilla_val, attn_train, attn_val):
    epochs = range(1, len(vanilla_train) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss plot
    axes[0].plot(epochs, vanilla_train, 'b--', label='Vanilla Train')
    axes[0].plot(epochs, vanilla_val,   'b-',  label='Vanilla Val')
    axes[0].plot(epochs, attn_train,    'r--', label='Attention Train')
    axes[0].plot(epochs, attn_val,      'r-',  label='Attention Val')
    axes[0].set_title('Training & Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Perplexity plot
    axes[1].plot(epochs, [math.exp(l) for l in vanilla_train], 'b--', label='Vanilla Train')
    axes[1].plot(epochs, [math.exp(l) for l in vanilla_val],   'b-',  label='Vanilla Val')
    axes[1].plot(epochs, [math.exp(l) for l in attn_train],    'r--', label='Attention Train')
    axes[1].plot(epochs, [math.exp(l) for l in attn_val],      'r-',  label='Attention Val')
    axes[1].set_title('Perplexity')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Perplexity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: learning_curves.png")


plot_learning_curves(
    vanilla_train_losses, vanilla_val_losses,
    attn_train_losses,    attn_val_losses
)

## Translation Inference

In [ ]:
def translate_sentence(model, sentence_str, src_vocab, trg_vocab, max_len=MAX_LEN):
    """
    Translates a single English sentence string.
    Returns:
        translated_tokens: list of French word strings
        attention_matrix:  (trg_len, src_len) numpy array, or None for vanilla
    """
    model.eval()

    # Tokenize & numericalize source
    tokens    = sentence_str.lower().split()
    src_ids   = [src_vocab.word2index.get(t, 3) for t in tokens] + [2]  # append <EOS>
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)           # (src_len, 1)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)

    dec_input     = torch.tensor([1]).to(device)   # <SOS>
    translated    = []
    attn_matrices = []

    with torch.no_grad():
        for _ in range(max_len):
            prediction, hidden, attn_weights = model.decoder(dec_input, hidden, encoder_outputs)
            top_token = prediction.argmax(1).item()

            if top_token == 2:   # <EOS>
                break

            translated.append(trg_vocab.index2word.get(top_token, '<UNK>'))

            if attn_weights is not None:
                attn_matrices.append(attn_weights.squeeze(0).cpu().numpy())

            dec_input = torch.tensor([top_token]).to(device)

    attention_matrix = np.stack(attn_matrices) if attn_matrices else None
    return translated, attention_matrix


print("Inference function ready.")

## BLEU Score Evaluation

In [ ]:
def compute_bleu(model, dataset, src_vocab, trg_vocab, n_samples=500):
    """
    Computes corpus BLEU on a random sample of the dataset.
    Uses smoothing to handle short sentences with 0-count n-grams.
    """
    hypotheses = []   # model translations
    references = []   # ground truth

    # Sample indices so we don't translate 4000 sentences
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))

    for i in indices:
        src_str, trg_str = pairs[dataset.indices[i]]
        translation, _   = translate_sentence(model, src_str, src_vocab, trg_vocab)

        hypotheses.append(translation)
        references.append([trg_str.split()])

    smoother = SmoothingFunction().method1
    bleu = corpus_bleu(references, hypotheses, smoothing_function=smoother)
    return bleu * 100   # return as percentage


# Load best checkpoints before evaluating
vanilla_model.load_state_dict(torch.load('vanilla_best.pt', map_location=device))
attn_model.load_state_dict(torch.load('attention_best.pt', map_location=device))

print("Computing BLEU scores on test set (this may take ~1 min)...")
vanilla_bleu = compute_bleu(vanilla_model, test_ds, src_vocab, trg_vocab)
attn_bleu    = compute_bleu(attn_model,    test_ds, src_vocab, trg_vocab)

print(f"\n{'='*40}")
print(f"  Vanilla Seq2Seq  BLEU: {vanilla_bleu:.2f}")
print(f"  Attention Seq2Seq BLEU: {attn_bleu:.2f}")
print(f"  Improvement:     +{attn_bleu - vanilla_bleu:.2f} BLEU points")
print(f"{'='*40}")

## Qualitative Analysis — Example Translations

In [ ]:
def show_translation_examples(n=6):
    """Print side-by-side examples: source | reference | vanilla | attention"""
    sample_indices = random.sample(range(len(test_ds)), n)

    print(f"{'SOURCE (EN)':<30} | {'REFERENCE (FR)':<30} | {'VANILLA':<25} | {'ATTENTION':<25}")
    print("-" * 120)

    for i in sample_indices:
        src_str, ref_str = pairs[test_ds.indices[i]]
        vanilla_trans, _ = translate_sentence(vanilla_model, src_str, src_vocab, trg_vocab)
        attn_trans, _    = translate_sentence(attn_model,    src_str, src_vocab, trg_vocab)

        print(f"{src_str:<30} | {ref_str:<30} | {' '.join(vanilla_trans):<25} | {' '.join(attn_trans):<25}")


show_translation_examples()

## Attention Visualization

In [ ]:
def plot_attention(src_str, trg_vocab, model, title="Attention", save_name=None):
    """
    Plots the attention weight matrix α for a single sentence.
    Rows = decoder output steps (French tokens generated)
    Cols = encoder input steps (English tokens)
    """
    translation, attention = translate_sentence(model, src_str, src_vocab, trg_vocab)

    if attention is None:
        print("This model has no attention weights to visualize.")
        return

    src_tokens = src_str.split() + ['<EOS>']
    trg_tokens = translation

    # attention shape: (trg_len, src_len)
    fig, ax = plt.subplots(figsize=(max(8, len(src_tokens)), max(6, len(trg_tokens))))

    cax = ax.matshow(attention, cmap='YlOrRd')
    fig.colorbar(cax)

    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(trg_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='left', fontsize=11)
    ax.set_yticklabels(trg_tokens, fontsize=11)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    ax.set_xlabel('Source (English)', fontsize=12)
    ax.set_ylabel('Translation (French)', fontsize=12)
    ax.set_title(f'{title}\n"{src_str}"\n→ "{" ".join(trg_tokens)}"', fontsize=12)

    plt.tight_layout()
    fname = save_name or f"attention_{src_str[:20].replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")


# --- Plot attention for 3 example sentences ---
example_sentences = [
    pairs[test_ds.indices[0]][0],
    pairs[test_ds.indices[1]][0],
    pairs[test_ds.indices[2]][0],
]

for i, sent in enumerate(example_sentences):
    plot_attention(
        sent, trg_vocab, attn_model,
        title=f"Bahdanau Attention Weights — Example {i+1}",
        save_name=f"attention_example_{i+1}.png"
    )

## Comparison Summary

In [ ]:
def print_comparison_summary():
    vanilla_test_loss = evaluate_epoch(vanilla_model, test_loader)
    attn_test_loss    = evaluate_epoch(attn_model,    test_loader)

    print("\n" + "="*55)
    print("          COMPARISON EXPERIMENT SUMMARY")
    print("="*55)
    print(f"{'Metric':<25} {'Vanilla':>12} {'Attention':>12}")
    print("-"*55)
    print(f"{'Test Loss':<25} {vanilla_test_loss:>12.4f} {attn_test_loss:>12.4f}")
    print(f"{'Test Perplexity':<25} {math.exp(vanilla_test_loss):>12.2f} {math.exp(attn_test_loss):>12.2f}")
    print(f"{'BLEU Score':<25} {vanilla_bleu:>11.2f}% {attn_bleu:>11.2f}%")
    print(f"{'BLEU Improvement':<25} {'':>12} {attn_bleu - vanilla_bleu:>+11.2f}%")
    print(f"{'Parameters':<25} {count_params(vanilla_model):>12,} {count_params(attn_model):>12,}")
    print("="*55)
    print()
    print("Key insight:")
    print("  Vanilla Seq2Seq compresses the entire source sentence into")
    print("  one fixed vector. Bahdanau attention lets the decoder look")
    print("  back at all encoder states, with learned weights — removing")
    print("  the information bottleneck and improving translation quality.")


print_comparison_summary()